# Text Classification con DistilRoBERTa + Optuna

Modelo alternativo al DistilBERT usando `distilroberta-base`.

**Instalación requerida:**
```
conda install datasets=="2.20.0"
conda install transformers=="4.40.1"
conda install numpy=="1.26.4"
```

In [1]:
# Data processing
import pandas as pd
import numpy as np
import os
import copy
import time
import datetime

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, cohen_kappa_score

from datasets import Dataset, DatasetDict

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

# Modeling
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    AdamW,
    get_scheduler
)

from tqdm.auto import tqdm
from utils import plot_confusion_matrix, get_artifact_filename
from joblib import load, dump

# Verificar CUDA
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

c:\Users\tomi_\anaconda3\envs\ldi2_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA disponible: True
GPU: NVIDIA GeForce RTX 3060


## Configuración

In [2]:
# Paths
BASE_DIR = '../'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

os.makedirs(PATH_TO_TEMP_FILES, exist_ok=True)
os.makedirs(PATH_TO_OPTUNA_ARTIFACTS, exist_ok=True)

# Parámetros
SEED = 42
TEST_SIZE = 0.2
BATCH_SIZE = 16

# Modelo
MODEL_CHECKPOINT = 'distilroberta-base'   # <-- único cambio respecto al notebook de DistilBERT
MODEL_NAME = '07 DistilRoBERTa'
MODEL_VERSION = '1.0'

print(f'Modelo: {MODEL_CHECKPOINT}')

Modelo: distilroberta-base


## Tokenizer

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f'Tokenizer cargado: {MODEL_CHECKPOINT}')

c:\Users\tomi_\anaconda3\envs\ldi2_cuda\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Tokenizer cargado: distilroberta-base


## Armado de Datasets

In [4]:
# Cargar datos
df = pd.read_csv(PATH_TO_TRAIN)
df = df[df['Description'].notnull()]
df['labels'] = df['AdoptionSpeed']

# Usar el mismo split del estudio LGB para mantener consistencia entre modelos
study_lgb = optuna.create_study(
    direction='maximize',
    storage="sqlite:///../work/db.sqlite3",
    study_name="04 - LGB Multiclass CV",
    load_if_exists=True
)

lgb_test_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS, get_artifact_filename(study_lgb, 'test')))

train_df = df[~df.PetID.isin(lgb_test_dataset.PetID)].reset_index(drop=True)
test_df  = df[df.PetID.isin(lgb_test_dataset.PetID)].reset_index(drop=True)

print(f'Train: {len(train_df)} registros | Test: {len(test_df)} registros')

# Convertir a HuggingFace Dataset
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'val':   Dataset.from_pandas(test_df)
})
dataset = dataset.class_encode_column('labels')

cols_to_remove = [col for col in dataset['train'].column_names if col != 'labels']
print(f'Columnas a remover en tokenización: {cols_to_remove}')

[I 2026-04-28 19:52:36,146] Using an existing study with name '04 - LGB Multiclass CV' instead of creating a new one.


Train: 11984 registros | Test: 2996 registros


Casting to class labels: 100%|██████████| 2996/2996 [00:00<00:00, 237859.83 examples/s]

Columnas a remover en tokenización: ['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed']


In [5]:
# Tokenización
def tokenize(batch):
    tok = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
    return tok(batch['Description'], padding=True, truncation=True, max_length=512)

dataset_enc = dataset.map(tokenize, batched=True, remove_columns=cols_to_remove, num_proc=1)
dataset_enc.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(dataset_enc['train'].column_names)

Map:   0%|          | 0/11984 [00:00<?, ? examples/s]c:\Users\tomi_\anaconda3\envs\ldi2_cuda\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 2996/2996 [00:01<00:00, 2212.04 examples/s]

['labels', 'input_ids', 'attention_mask']


In [6]:
# Data collator y DataLoaders
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_dataloader = DataLoader(
    dataset_enc['train'], shuffle=True, batch_size=BATCH_SIZE, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    dataset_enc['val'], batch_size=BATCH_SIZE, collate_fn=data_collator
)

# IDs del test set para el DataFrame de predicciones
test_sample_ids = list(test_df.PetID)

print(f'Batches de train: {len(train_dataloader)} | Batches de val: {len(eval_dataloader)}')

Batches de train: 749 | Batches de val: 188


## Artifact Store de Optuna

In [7]:
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)

## Función de Entrenamiento

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

num_labels = dataset['train'].features['labels'].num_classes
print(f'Número de clases: {num_labels}')

Device: cuda
Número de clases: 5


In [9]:
def train_val(model, dataloaders, datasets, device, num_epochs=4, lr=2e-5, trial=None):

    since = time.time()

    optimizer = AdamW(model.parameters(), lr=lr)

    num_training_steps = num_epochs * len(dataloaders['train'])
    lr_scheduler = get_scheduler(
        'linear',
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps
    )

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc   = 0.0
    best_kappa = -999

    try:
        previous_best = study.best_value
    except:
        previous_best = -999

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        kappa_labels_true      = []
        kappa_labels_predicted = []
        output_scores          = []

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss     = 0.0
            running_corrects = 0

            for batch in tqdm(dataloaders[phase]):
                batch  = batch.to(device)
                labels = batch.labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs     = model(**batch)
                    loss        = outputs.loss
                    preds       = torch.nn.functional.softmax(outputs.logits, dim=-1)
                    preds_labels = torch.argmax(preds, dim=-1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                        lr_scheduler.step()
                    elif phase == 'val':
                        kappa_labels_true.extend(labels.cpu().numpy().tolist())
                        kappa_labels_predicted.extend(preds_labels.cpu().numpy().tolist())
                        outputs_np = preds.cpu().numpy()
                        output_scores.extend([outputs_np[i, :] for i in range(outputs_np.shape[0])])

                running_loss     += loss.item() * labels.size(0)
                running_corrects += torch.sum(preds_labels == labels.data)

            epoch_loss = running_loss / len(datasets[phase])
            epoch_acc  = running_corrects.double() / len(datasets[phase])

            if phase == 'train':
                kappa_score = float('nan')
            else:
                kappa_score = cohen_kappa_score(
                    kappa_labels_true,
                    kappa_labels_predicted,
                    weights='quadratic'
                )

            print(f'{phase.title()} Loss: {epoch_loss:.4f} Acc: {epoch_acc*100:.2f}% Kappa: {kappa_score:.3f}')

            if phase == 'val' and kappa_score > best_kappa:
                best_acc   = epoch_acc
                best_kappa = kappa_score
                best_model_wts = copy.deepcopy(model.state_dict())

                if trial is not None and best_kappa > previous_best:
                    # Guardar predicciones del test
                    predicted_filename = os.path.join(
                        PATH_TO_TEMP_FILES,
                        f'test_{trial.study.study_name}_{trial.number}.joblib'
                    )
                    predicted_df = pd.DataFrame({
                        'PetID': test_sample_ids,
                        'pred':  output_scores
                    }).merge(test_df, on='PetID')
                    dump(predicted_df, predicted_filename)

                    # Guardar confusion matrix
                    cm_filename = os.path.join(
                        PATH_TO_TEMP_FILES,
                        f'cm_{trial.study.study_name}_{trial.number}.jpg'
                    )
                    plot_confusion_matrix(kappa_labels_true, kappa_labels_predicted).write_image(cm_filename)

    time_elapsed = time.time() - since
    print(f'Training completo en {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Mejor val Acc: {best_acc * 100:.2f}%  |  Mejor Kappa: {best_kappa:.4f}')

    model.load_state_dict(best_model_wts)

    if trial is not None and best_kappa > previous_best:
        upload_artifact(trial, predicted_filename, artifact_store)
        upload_artifact(trial, cm_filename, artifact_store)

        model_path = os.path.join(PATH_TO_TEMP_FILES, f'{MODEL_NAME}_{MODEL_VERSION}_{trial.number}.pth')
        torch.save(model, model_path)
        upload_artifact(trial, model_path, artifact_store)

    return model, best_kappa

## Función Objetivo de Optuna

In [10]:
def optuna_train(trial):
    # Hiperparámetros a optimizar
    num_epochs = trial.suggest_int('epochs', 1, 4)
    lr         = trial.suggest_float('lr', 1e-5, 5e-5, log=True)

    print(f'\n--- Trial {trial.number} | epochs={num_epochs} | lr={lr:.2e} ---')

    # Cargar modelo fresco en cada trial
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=num_labels
    ).to(device)

    dataloaders = {'train': train_dataloader, 'val': eval_dataloader}
    datasets    = {'train': dataset_enc['train'], 'val': dataset_enc['val']}

    _, best_kappa = train_val(
        model=model,
        dataloaders=dataloaders,
        datasets=datasets,
        device=device,
        num_epochs=num_epochs,
        lr=lr,
        trial=trial
    )

    return best_kappa

## Optimización con Optuna

In [11]:
study = optuna.create_study(
    direction='maximize',
    storage='sqlite:///../work/db.sqlite3',
    study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
    load_if_exists=True
)

study.optimize(optuna_train, n_trials=5)

print(f'\n=== Resultado final ===')
print(f'Mejor trial: #{study.best_trial.number}')
print(f'Mejor kappa: {study.best_value:.4f}')
print(f'Mejores parámetros: {study.best_params}')

[I 2026-04-28 19:53:11,659] A new study created in RDB with name: 07 DistilRoBERTa_1.0



--- Trial 0 | epochs=4 | lr=4.97e-05 ---


c:\Users\tomi_\anaconda3\envs\ldi2_cuda\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\tomi_\anaconda3\envs\ldi2_cuda\Lib\site-packages\transformers\optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 0/3
----------


100%|██████████| 749/749 [06:27<00:00,  1.93it/s]


...

Val Loss: 1.4269 Acc: 36.32% Kappa: 0.225
Epoch 3/3
----------


100%|██████████| 749/749 [07:09<00:00,  1.75it/s]


Train Loss: 1.1710 Acc: 49.84% Kappa: nan


100%|██████████| 188/188 [00:35<00:00,  5.28it/s]
[I 2026-04-28 21:49:55,522] Trial 4 finished with value: 0.23276178395239855 and parameters: {'epochs': 4, 'lr': 2.342538502652576e-05}. Best is trial 0 with value: 0.23823941501861212.


Val Loss: 1.4838 Acc: 36.55% Kappa: 0.233
Training completo en 30m 53s
Mejor val Acc: 36.55%  |  Mejor Kappa: 0.2328

=== Resultado final ===
Mejor trial: #0
Mejor kappa: 0.2382
Mejores parámetros: {'epochs': 4, 'lr': 4.966766434523626e-05}


## Exportar Scores del Mejor Trial

In [12]:
# Recuperar el mejor trial
best_trial = study.best_trial
print(f'Mejor trial: #{best_trial.number} | Kappa: {best_trial.value:.4f}')
print(f'Parámetros: {best_trial.params}')

# Cargar predicciones del mejor trial
predicted_filename = os.path.join(
    PATH_TO_TEMP_FILES,
    f'test_{study.study_name}_{best_trial.number}.joblib'
)
predicted_df = load(predicted_filename)

# Expandir probabilidades en columnas separadas
scores = pd.DataFrame(
    predicted_df['pred'].tolist(),
    columns=[f'distilroberta_prob_{i}' for i in range(5)]
)

output_df = pd.concat(
    [predicted_df[['PetID', 'AdoptionSpeed']].reset_index(drop=True), scores],
    axis=1
)

print(f'Shape del output: {output_df.shape}')
print(output_df.head())

# Exportar CSV
output_path = os.path.join(BASE_DIR, 'output', 'scores_distilroberta.csv')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
output_df.to_csv(output_path, index=False)

print(f'\n✓ Scores exportados a: {output_path}')

Mejor trial: #0 | Kappa: 0.2382
Parámetros: {'epochs': 4, 'lr': 4.966766434523626e-05}
Shape del output: (2996, 7)
       PetID  AdoptionSpeed  distilroberta_prob_0  distilroberta_prob_1  \
0  8e76c8e39              1              0.020085              0.231145   
1  6436c1a59              2              0.011618              0.082706   
2  988988d5b              1              0.056191              0.226916   
3  efbf1703a              2              0.020375              0.228862   
4  543130f60              4              0.014998              0.417551   

   distilroberta_prob_2  distilroberta_prob_3  distilroberta_prob_4  
0              0.293072              0.189136              0.266561  
1              0.325336              0.320968              0.259372  
2              0.218030              0.135130              0.363733  
3              0.247505              0.163621              0.339636  
4              0.347381              0.148833              0.071237  

✓ Scores expo